#### Import libraries

In [0]:

from pyspark.sql import DataFrame
from pyspark.sql import functions as SQL_FUNCTIONS

import matplotlib.pyplot as plt
import pandas as pd

#### Create useful variables and read the data table

In [0]:

dbutils.widgets.text("silver_table", "workspace.bda_taxi.taxi_silver")

silver_table = dbutils.widgets.get("silver_table").strip()
print("Silver table:", silver_table)

# Load Silver data
silver_dataframe = spark.table(silver_table)
print("Silver rows:", silver_dataframe.count())
display(silver_dataframe.limit(5))

#### Helper functions for EDA: graphs etc.

In [0]:

def plot_histogram(
    dataframe: DataFrame,
    column_name: str,
    bins: int = 50,
    log_scale: bool = False,
    title_suffix: str = ""
) -> None:
    """
    Plots a histogram for a numeric column using Pandas + Matplotlib.
    Intended for distribution inspection and outlier analysis.
    """
    pdf = dataframe.select(column_name).dropna().toPandas()

    plt.figure(figsize=(8, 5))
    plt.hist(pdf[column_name], bins=bins)
    if log_scale:
        plt.yscale("log")
    plt.xlabel(column_name)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {column_name} {title_suffix}")
    plt.tight_layout()
    plt.show()


def show_quantiles(
    dataframe: DataFrame,
    column_name: str,
    quantiles: list = [0.5, 0.9, 0.95, 0.99, 0.995]
) -> None:
    """
    Displays approximate quantiles for a numeric column.
    Useful for justifying outlier thresholds.
    """
    values = dataframe.approxQuantile(column_name, quantiles, 0.01)
    for q, v in zip(quantiles, values):
        print(f"{column_name} quantile {q}: {v}")


def top_k_table_with_count(
    dataframe: DataFrame,
    group_column: str,
    metric_column: str,
    k: int = 10,
    metric_name: str = "avg_value",
    MIN_TRIPS: int = 100
) -> DataFrame:
    """
    Returns a top-k table by average metric, including trip count for context.
    """
    return (
        dataframe
        .groupBy(group_column)
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(metric_column).alias(metric_name)
        )
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .orderBy(SQL_FUNCTIONS.desc(metric_name))
        .limit(k)
    )

#### Plot the charts and list out the data to allow examination

##### Distribution analysis and outlier justification

In [0]:
plot_histogram(
    silver_dataframe,
    column_name="trip_distance",
    bins=10,
    log_scale=False,
    title_suffix="(linear scale)"
)

# Trip distance distribution
plot_histogram(
    silver_dataframe,
    column_name="trip_distance",
    bins=60,
    log_scale=True,
    title_suffix="(log-scaled)"
)

show_quantiles(silver_dataframe, "trip_distance")

# Fare amount distribution
plot_histogram(
    silver_dataframe,
    column_name="fare_amount",
    bins=60,
    log_scale=True,
    title_suffix="(log-scaled)"
)

show_quantiles(silver_dataframe, "fare_amount")

##### Temporal analysis

In [0]:
# Trips by time bucket
trips_by_time_bucket = (
    silver_dataframe
    .groupBy("time_bucket")
    .count()
    .orderBy("time_bucket")
)

display(trips_by_time_bucket)


# Average fare by time bucket
avg_fare_by_time_bucket = (
    silver_dataframe
    .groupBy("time_bucket")
    .agg(SQL_FUNCTIONS.avg("fare_amount").alias("avg_fare"))
    .orderBy("time_bucket")
)

display(avg_fare_by_time_bucket)

##### Tipphing behaviour (credit card trips only)

In [0]:
# Restrict to trips where tips are recorded
tips_recorded_df = silver_dataframe.filter(SQL_FUNCTIONS.col("tip_recorded") == 1)

print("Trips with recorded tips:", tips_recorded_df.count())


# Tip rate by time bucket
tip_rate_by_time_bucket = (
    tips_recorded_df
    .groupBy("time_bucket")
    .agg(SQL_FUNCTIONS.count("*").alias("trip_count"), SQL_FUNCTIONS.avg("tipped").alias("tip_rate"))
    .orderBy("time_bucket")
)

display(tip_rate_by_time_bucket)


# Average tip amount by time bucket
avg_tip_by_time_bucket = (
    tips_recorded_df
    .groupBy("time_bucket")
    .agg(SQL_FUNCTIONS.count("*").alias("trip_count"), SQL_FUNCTIONS.avg("tip_amount").alias("avg_tip"))
    .orderBy("time_bucket")
)

display(avg_tip_by_time_bucket)

##### Spatial analysis (pickup and dropoff zones)

In [0]:
# Top pickup zones by average fare
top_pickup_fare = top_k_table_with_count(
    silver_dataframe,
    group_column="PU_Zone",
    metric_column="fare_amount",
    k=10,
    metric_name="avg_fare"
)

display(top_pickup_fare)

# Top dropoff zones by average fare
top_dropoff_fare = top_k_table_with_count(
    silver_dataframe,
    group_column="DO_Zone",
    metric_column="fare_amount",
    k=10,
    metric_name="avg_fare"
)

display(top_dropoff_fare)

# Top pickup zones by average tip (credit card only)
top_pickup_tip = top_k_table_with_count(
    tips_recorded_df,
    group_column="PU_Zone",
    metric_column="tip_amount",
    k=10,
    metric_name="avg_tip"
)

display(top_pickup_tip)

# Top dropoff zones by average tip (credit card only)
top_dropoff_tip = top_k_table_with_count(
    tips_recorded_df,
    group_column="DO_Zone",
    metric_column="tip_amount",
    k=10,
    metric_name="avg_tip"
)

display(top_dropoff_tip)


#### [Extra] Overlay of datasets on real map of NYC with taxi zones

In [0]:
import os
import json
from typing import Dict, List, Optional, Tuple

import pandas as pd
import plotly.express as px
from pyspark.sql import DataFrame
from pyspark.sql import functions as SQL_FUNCTIONS


# GeoJSON loading helpers

def load_geojson_feature_collection(geojson_path: str) -> dict:
    """
    Loads a GeoJSON FeatureCollection from a local repo path.
    """
    with open(geojson_path, "r") as file_handle:
        return json.load(file_handle)


def detect_location_id_property(geojson_fc: dict) -> str:
    """
    Detect the property name inside each GeoJSON feature that contains the taxi zone LocationID.
    Common names: 'LocationID', 'locationid', 'location_id'.
    """
    properties = geojson_fc["features"][0]["properties"]
    candidates = ["LocationID", "locationid", "location_id", "LOCATIONID", "OBJECTID"]
    for candidate in candidates:
        if candidate in properties:
            return candidate
    raise ValueError(
        f"Could not find a LocationID property in GeoJSON. Available keys: {list(properties.keys())}"
    )


def ensure_join_key_as_string(pdf: pd.DataFrame, join_col: str) -> pd.DataFrame:
    """
    Ensures join column is string typed for consistent matching with GeoJSON properties.
    """
    pdf = pdf.copy()
    pdf[join_col] = pdf[join_col].astype("int64").astype(str)
    return pdf


def compute_geojson_match_stats(geojson_fc: dict, geojson_location_prop: str, pdf: pd.DataFrame, pdf_join_col: str) -> None:
    """
    Prints simple match statistics to confirm that dataframe join keys exist in GeoJSON.
    """
    geo_ids = set(str(feat["properties"].get(geojson_location_prop)) for feat in geojson_fc["features"])
    df_ids = set(pdf[pdf_join_col].unique())

    matches = len(geo_ids.intersection(df_ids))
    print("GeoJSON zones:", len(geo_ids))
    print("Data zones:", len(df_ids))
    print("Matches:", matches)



# Aggregation helpers

def build_zone_metric_dataframe(
    silver_df: DataFrame,
    zone_side: str,
    metric: str,
    aggregation: str,
    credit_card_only: bool = False
) -> Tuple[pd.DataFrame, str, str]:
    """
    Aggregates a metric by taxi zone (pickup or dropoff), returning a Pandas DataFrame ready to map.

    Parameters
    ----------
    zone_side:
        'pickup' -> uses PULocationID
        'dropoff' -> uses DOLocationID
    metric:
        One of: 'fare_amount', 'tip_amount', 'total_amount', 'trip_distance', 'tipped'
    aggregation:
        'avg' or 'sum' or 'median' (count always included as 'trip_count')
    credit_card_only:
        If True, filter to tip_recorded == 1 (recommended for tip-based metrics)

    Returns
    -------
    (pdf, join_col_str, metric_col_name)
        pdf: includes join key as string, trip_count, and computed metric column
        join_col_str: column name to use in plotly 'locations'
        metric_col_name: column name to use for plotly color
    """
    zone_side = zone_side.strip().lower()
    if zone_side not in ["pickup", "dropoff"]:
        raise ValueError("zone_side must be 'pickup' or 'dropoff'")

    zone_id_col = "PULocationID" if zone_side == "pickup" else "DOLocationID"
    join_col_str = f"{zone_id_col}_str"

    if credit_card_only:
        df = silver_df.filter(SQL_FUNCTIONS.col("tip_recorded") == 1)
    else:
        df = silver_df

    if aggregation == "avg":
        metric_expr = SQL_FUNCTIONS.avg(metric).alias(f"avg_{metric}")
        metric_col_name = f"avg_{metric}"
    elif aggregation == "sum":
        metric_expr = SQL_FUNCTIONS.sum(metric).alias(f"sum_{metric}")
        metric_col_name = f"sum_{metric}"
    elif aggregation == "median":
        metric_expr = SQL_FUNCTIONS.median(metric).alias(f"median_{metric}")
        metric_col_name = f"median_{metric}"
    else:
        raise ValueError("aggregation must be 'avg', 'sum' or 'median")

    agg_df = (
        df.groupBy(zone_id_col)
          .agg(
              SQL_FUNCTIONS.count("*").alias("trip_count"),
              metric_expr
          )
    )

    pdf = agg_df.toPandas()

    # Ensure key is string for GeoJSON matching
    pdf[join_col_str] = pdf[zone_id_col].astype("int64").astype(str)

    return pdf, join_col_str, metric_col_name


# Plotting helpers

def plot_zone_choropleth(
    pdf: pd.DataFrame,
    geojson_fc: dict,
    geojson_location_prop: str,
    locations_col: str,
    color_col: str,
    title: str,
    hover_columns: Optional[List[str]] = None,
    zoom: float = 9.5,
    center: Optional[dict] = None
):
    """
    Plots a taxi zone choropleth with readable hover tooltips.

    hover_columns:
        Columns to show in tooltip. Use friendly names by renaming pdf columns beforehand.
    """
    if center is None:
        center = {"lat": 40.73, "lon": -73.94}

    # Default hover columns: always include trip_count and the mapped color
    if hover_columns is None:
        hover_columns = []
        if "trip_count" in pdf.columns:
            hover_columns.append("trip_count")
        if color_col in pdf.columns and color_col not in hover_columns:
            hover_columns.append(color_col)

    # Build hover data dict to control formatting and visibility
    hover_data = {}
    for col in pdf.columns:
        hover_data[col] = False  # hide everything by default

    # Show desired columns with formatting:
    for col in hover_columns:
        if col in pdf.columns:
            # Provide nicer formatting for numeric columns
            if pd.api.types.is_numeric_dtype(pdf[col]):
                hover_data[col] = ":.2f" if col != "trip_count" else True
            else:
                hover_data[col] = True

    fig = px.choropleth_mapbox(
        pdf,
        geojson=geojson_fc,
        locations=locations_col,
        featureidkey=f"properties.{geojson_location_prop}",
        color=color_col,
        mapbox_style="carto-positron",
        zoom=zoom,
        center=center,
        opacity=0.65,
        title=title,
        hover_data=hover_data
    )

    # Make polygons visible
    fig.update_traces(marker_line_width=1.0)
    fig.update_traces(marker_line_color="black")

    # Improve layout readability
    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        title=dict(x=0.01),
    )

    fig.show()


In [0]:
# Load GeoJSON from repo
repo_root = os.getcwd()
geojson_path = os.path.join(repo_root, "data", "NYC_Taxi_Zones_20260206.geojson")

taxi_zones_geojson = load_geojson_feature_collection(geojson_path)
geojson_location_prop = detect_location_id_property(taxi_zones_geojson)

print("Using GeoJSON LocationID property:", geojson_location_prop)

In [0]:

# Average tip amount by PICKUP zone (credit card only)
pickup_tip_pdf, pickup_locations_col, pickup_color_col = build_zone_metric_dataframe(
    silver_df=silver_dataframe,
    zone_side="pickup",
    metric="tip_amount",
    aggregation="avg",
    credit_card_only=True
)

compute_geojson_match_stats(
    geojson_fc=taxi_zones_geojson,
    geojson_location_prop=geojson_location_prop,
    pdf=pickup_tip_pdf,
    pdf_join_col=pickup_locations_col
)

# Rename columns for friendlier tooltip labels
pickup_tip_pdf = pickup_tip_pdf.rename(columns={
    pickup_color_col: "Average Tip ($)"
})

plot_zone_choropleth(
    pdf=pickup_tip_pdf,
    geojson_fc=taxi_zones_geojson,
    geojson_location_prop=geojson_location_prop,
    locations_col=pickup_locations_col,
    color_col="Average Tip ($)",
    title="Average Tip Amount by Pick-up Taxi Zone (Credit Card Trips)",
    hover_columns=["trip_count", "Average Tip ($)"]
)





In [0]:
# Median fare amount by PICKUP zone (credit card only)
dropoff_fare_pdf, dropoff_locations_col, dropoff_color_col = build_zone_metric_dataframe(
    silver_df=silver_dataframe,
    zone_side="pickup",
    metric="fare_amount",
    aggregation="median",
    credit_card_only=False
)

compute_geojson_match_stats(
    geojson_fc=taxi_zones_geojson,
    geojson_location_prop=geojson_location_prop,
    pdf=dropoff_fare_pdf,
    pdf_join_col=dropoff_locations_col
)

dropoff_fare_pdf = dropoff_fare_pdf.rename(columns={
    dropoff_color_col: "Median Fare ($)"
})

plot_zone_choropleth(
    pdf=dropoff_fare_pdf,
    geojson_fc=taxi_zones_geojson,
    geojson_location_prop=geojson_location_prop,
    locations_col=dropoff_locations_col,
    color_col="Median Fare ($)",
    title="Median Fare Amount by Pick-up Taxi Zone (All Trips - All Payment types)",
    hover_columns=["trip_count", "Median Fare ($)"]
)

In [0]:
# Average fare amount by DROPOFF zone (all payment types)
dropoff_fare_pdf, dropoff_locations_col, dropoff_color_col = build_zone_metric_dataframe(
    silver_df=silver_dataframe,
    zone_side="dropoff",
    metric="fare_amount",
    aggregation="avg",
    credit_card_only=False
)

compute_geojson_match_stats(
    geojson_fc=taxi_zones_geojson,
    geojson_location_prop=geojson_location_prop,
    pdf=dropoff_fare_pdf,
    pdf_join_col=dropoff_locations_col
)

dropoff_fare_pdf = dropoff_fare_pdf.rename(columns={
    dropoff_color_col: "Average Fare ($)"
})

plot_zone_choropleth(
    pdf=dropoff_fare_pdf,
    geojson_fc=taxi_zones_geojson,
    geojson_location_prop=geojson_location_prop,
    locations_col=dropoff_locations_col,
    color_col="Average Fare ($)",
    title="Average Fare Amount by Drop-off Taxi Zone (All Trips - All Payment types)",
    hover_columns=["trip_count", "Average Fare ($)"]
)

In [0]:
# Trip volume by PICKUP zone
pickup_volume_pdf, pickup_locations_col, _ = build_zone_metric_dataframe(
    silver_df=silver_dataframe,
    zone_side="pickup",
    metric="fare_amount",    # metric unused for count, but kept for consistent function signature
    aggregation="avg",
    credit_card_only=False
)

plot_zone_choropleth(
    pdf=pickup_volume_pdf,
    geojson_fc=taxi_zones_geojson,
    geojson_location_prop=geojson_location_prop,
    locations_col=pickup_locations_col,
    color_col="trip_count",
    title="Trip Volume by Pick-up Taxi Zone (All Trips - All Payment types)",
    hover_columns=["trip_count"]
)